# Arbeid

## Doelen

Nu dat we de microscopische grootheden van de moleculen hebben verbonden aan de macroscopische grootheden van het gas, kunnen we de thermodynamica van het gas op een andere manier bestuderen. In dit werkblad gaan we kijken hoe de temperatuur en de druk veranderen onder invloed van een zuiger die het volume verandert. 

Eerst herhalen we de delen van de code die we nodig hebben:

- klasse voor het deeltje met bijbehorende functies
- variabelen en randcondities van controle volume
- functies voor een lijst deeltjes

Daarna voegen we code toe voor de dynamiek van de zuiger:

- zuiger implementeren in volume en dynamische formules
- bestuderen van temperatuur en druk als functie van volume

## Laden van eerdere code

We beginnen weer met de noodzakelijke pakketten en de constanten:


(Tekst Freek: 10 deeltjes, met bewegend wand, eerst isotherm (snelheid van deeltjes niet gewijzigd) - snelheid van deeltjes verandert wel door beweging vd wand. p(t) en p(V))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

BOX_SIZE_0 = 10                # Hoogte en breedte startvolume
BOX_LENGTH_0 = BOX_SIZE_0 / 2
N = 40                         # Aantal deeltjes
V_0 = 1                        # Startsnelheid van deeltjes
RADIUS = 0.3                   # Straal van moleculen
DT = 0.1 * RADIUS / V_0        # Tijdstap om geen botsing te missen

De klasse voor de gasmoleculen en de functies voor hun onderlinge interactie:

In [ ]:
class ParticleClass:
    def __init__(self, m, v, r, R):
        """ maakt een deeltje (constructor) """
        self.m = m                         
        self.v = np.array(v, dtype=float)  
        self.r = np.array(r, dtype=float)  
        self.R = R

    def update_position(self):
        """ verandert positie voor één tijdstap """
        self.r += self.v * DT 
            
    @property
    def momentum(self):
        return self.m * self.v
    
    @property
    def kin_energy(self):
        return 1/2 * self.m * np.dot(self.v, self.v)
    
def collide_detection(p1: ParticleClass, p2: ParticleClass) -> bool:
    """ Geeft TRUE als de deeltjes overlappen """
    return np.linalg.norm(p1.r - p2.r) < (p1.R + p2.R)


def particle_collision(p1: ParticleClass, p2: ParticleClass):
    """ past snelheden aan uitgaande van overlap """
    r1, r2 = p1.r, p2.r
    v1, v2 = p1.v, p2.v
    m1, m2 = p1.m, p2.m
    delta_r = r1 - r2
    delta_v = v1 - v2
    # Als deeltjes van elkaar weg bewegen dan geen botsing
    if np.dot(delta_r, delta_v) > 0:
        return
    distance_squared = np.dot(delta_r, delta_r) + 1e-12  # voorkom deling door 0
    # Botsing oplossen volgens elastische botsing in 2D
    v1_new = v1 - 2 * m2 / (m1 + m2) * np.dot(delta_v, delta_r) / distance_squared * delta_r
    v2_new = v2 - 2 * m1 / (m1 + m2) * np.dot(-delta_v, -delta_r) / distance_squared * (-delta_r)
    p1.v = v1_new
    p2.v = v2_new

Het volume en de randvoorwaarden zullen we moeten aanpassen aan de nieuwe situatie. Het volume zal nu niet meer altijd een vierkant zijn van `BOX_SIZE_0` bij `BOX_SIZE_0`. Laten we aannemen dat de zuiger altijd in de horizontale richting verplaatst en het volume symmetrisch houdt ten opzichte van de oorsprong. Er is dus een zuiger aan de linker wand die een tegengestelde verplaatsing heeft aan die in de rechter wand. 

We maken eerst een aantal variabelen aan die bij het volume horen:

In [ ]:
box_height = BOX_SIZE_0
box_length = BOX_SIZE_0
pressure = 0.0

De functies die bij het volume en de randvoorwaarden horen moeten we een klein beetje aanpassen, zodat we niet langer uitgaan van de constante waarde van de lengte en hoogte. Om de variabelen `box_height` en `box_length` die we hierboven gedefinieerd hebben, later in functies te gebruiken, moeten we ze telkens oproepen met het keyword `global`. Dit is hieronder uitgewerkt. 

In [ ]:
def box_collision(particle: ParticleClass):
    global box_length, box_height
    """ botsing met harde wanden """
    if abs(particle.r[0]) + particle.R > box_length: 
        particle.v[0] = -particle.v[0]                                       # Omdraaien van de snelheid
        particle.r[0] = np.sign(particle.r[0]) * (box_length - particle.R)  # Zet terug net binnen box                 
    if abs(particle.r[1]) + particle.R > box_height: 
        particle.v[1] = -particle.v[1]     
        particle.r[1] = np.sign(particle.r[1]) * (box_height - particle.R) 

En dan laden we ook alle functies die over de gehele lijst met deeltjes werken. Let op dat we ook hier de nodige variabelen met `global` moeten aanroepen.

In [ ]:
def create_particles(particles):
    """ Leegmaken en opnieuw aanmaken van deeltjes  in lijst """
    global box_length, box_height
    particles.clear()
    for i in range(N):
        vx = np.random.uniform(-V_0, V_0)
        vy = np.random.choice([-1, 1]) * np.sqrt(V_0**2 - vx**2)        
        x = np.random.uniform(-box_length+RADIUS, box_length-RADIUS, 1)
        y = np.random.uniform(-box_height+RADIUS, box_height-RADIUS, 1)
        particles.append(ParticleClass(m=1.0, v=[vx, vy], r=[x, y], R=RADIUS))
        
def handle_collisions(particles):
    """ alle onderlinge botsingen afhandelen voor deeltjes in lijst """
    ignore_list = []
    for p1 in particles:
        if p1 in ignore_list:                                   # overslaan als partner al geweest is
            continue
        for p2 in particles:
            if p1 is p2:
                continue
            if collide_detection(p1, p2):
                particle_collision(p1, p2)
                ignore_list.append(p2)                          # partner uitsluiten van de berekening

def handle_walls(particles):
    """ botsing met wanden controleren voor alle deeltjes in lijst en bepaling druk """
    global pressure, box_length, box_height     # om pressure buiten de functie te kunnen gebruiken
    total_impulse = 0.0                         # totale stoot op wanden
    for p in particles:
        momentum_old = p.momentum
        box_collision(p)
        impulse = p.momentum - momentum_old
        total_impulse += np.linalg.norm(impulse)**0.5 # stoot is altijd netjes loodrecht naar buiten
    pressure = total_impulse / ((2 * box_length + 2 * box_height) * DT)  # omtrek volume is oppervlak (2D sim)

def take_time_step(particles):
    """ zet tijdstap voor een lijst deeltjes en verwerk alle botsingen onderling en met wanden """
    for p in particles:
        p.update_position()
    handle_walls(particles)  
    handle_collisions(particles)

## Implementeren (dubbele) zuiger

